In [1]:
# ===== Cell 1: 强制只使用 GPU 0（必须放在最前面） =====
import os

# 这两行必须在 import torch / import nichecompass 之前
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
print("现在 notebook 只会看到 1 块 GPU，并且它对应物理 GPU 0")

CUDA_VISIBLE_DEVICES = 0
现在 notebook 只会看到 1 块 GPU，并且它对应物理 GPU 0


In [2]:
# ===== Cell 2: 验证 GPU 是否锁定成功 =====
import sys
import torch

LOCAL_SRC = "/home/zhangjunyi/xiangmu/nichecompass-main/src"
if LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)

import nichecompass as nc

print("nichecompass loaded from:")
print(nc.__file__)

print("\ntorch.cuda.is_available() =", torch.cuda.is_available())
print("torch.cuda.device_count() =", torch.cuda.device_count())

if torch.cuda.is_available():
    print("当前可见 GPU 名称 =", torch.cuda.get_device_name(0))
    print("当前可见 GPU 编号 = 0")
else:
    print("没有检测到 GPU，说明前面环境或内核有问题。")

assert nc.__file__.startswith("/home/zhangjunyi/xiangmu/nichecompass-main/src"), \
    "当前导入的不是你本地修改后的 NicheCompass，请先重启 kernel 再运行。"

nichecompass loaded from:
/home/zhangjunyi/xiangmu/nichecompass-main/src/nichecompass/__init__.py

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
当前可见 GPU 名称 = A100 80GB PCIe
当前可见 GPU 编号 = 0


Cell 1：确保导入的是你本地修改版 NicheCompass

In [3]:
# ===== Cell 1: 导入你本地修改后的 NicheCompass =====
import os
import sys

LOCAL_SRC = "/home/zhangjunyi/xiangmu/nichecompass-main/src"

# 强制把本地源码放最前面
if LOCAL_SRC not in sys.path:
    sys.path.insert(0, LOCAL_SRC)

# 如果你之前导入过旧版，建议先重启 kernel 再运行本 cell
import nichecompass as nc
print("nichecompass loaded from:")
print(nc.__file__)

assert nc.__file__.startswith("/home/zhangjunyi/xiangmu/nichecompass-main/src"), \
    "当前导入的不是你本地修改后的 NicheCompass，请先重启 kernel。"

nichecompass loaded from:
/home/zhangjunyi/xiangmu/nichecompass-main/src/nichecompass/__init__.py


Cell 2：导入依赖 + 定义四元组

In [4]:
# ===== Cell 2: 依赖 + 四元组 =====
import warnings
warnings.filterwarnings("ignore")

from io import StringIO
from pathlib import Path

import decoupler as dc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import squidpy as sq
import torch
from sklearn.preprocessing import MinMaxScaler

tmcn_csv = """TMCN_Name,Source_Pathways,Source_Genes,Target_Genes,Biologic_Meaning
TMCN_Lactate_Axis,"EGFR,PI3K,Hypoxia","HK2,LDHA,LDHB,SLC16A3","SLC16A1,HCAR1",乳酸_肿瘤酸化与代谢重生
TMCN_Adenosine_Axis,"Hypoxia","MYC,ENTPD1,NT5E","ADORA1,ADORA2A,ADORA2B,ADORA3",腺苷_强效免疫抑制与血管生成
TMCN_PGE2_Axis,"NFkB,JAK-STAT","PTGS2,PTGES","PTGER1,PTGER2,PTGER4",前列腺素E2_炎症微环境重塑
TMCN_Glutamine_Axis,"JAK-STAT,TGFb","MYC,GLUL","SLC1A5,SLC38A2",谷氨酰胺_基质-肿瘤代谢共生(寄生)
TMCN_Succinate_Axis,"Hypoxia","EGLN1,TET2,SLC25A10","SUCNR1",琥珀酸_琥珀酸介导的巨噬细胞极化
"""

tmcn_df = pd.read_csv(StringIO(tmcn_csv))
display(tmcn_df)

AXES = [x.replace("TMCN_", "").replace("_Axis", "") for x in tmcn_df["TMCN_Name"].tolist()]
print("Axes:", AXES)

,TMCN_Name,Source_Pathways,Source_Genes,Target_Genes,Biologic_Meaning
0,TMCN_Lactate_Axis,"EGFR,PI3K,Hypoxia","HK2,LDHA,LDHB,SLC16A3","SLC16A1,HCAR1",乳酸_肿瘤酸化与代谢重生
1,TMCN_Adenosine_Axis,Hypoxia,"MYC,ENTPD1,NT5E","ADORA1,ADORA2A,ADORA2B,ADORA3",腺苷_强效免疫抑制与血管生成
2,TMCN_PGE2_Axis,"NFkB,JAK-STAT","PTGS2,PTGES","PTGER1,PTGER2,PTGER4",前列腺素E2_炎症微环境重塑
3,TMCN_Glutamine_Axis,"JAK-STAT,TGFb","MYC,GLUL","SLC1A5,SLC38A2",谷氨酰胺_基质-肿瘤代谢共生(寄生)
4,TMCN_Succinate_Axis,Hypoxia,"EGLN1,TET2,SLC25A10",SUCNR1,琥珀酸_琥珀酸介导的巨噬细胞极化


Axes: ['Lactate', 'Adenosine', 'PGE2', 'Glutamine', 'Succinate']


Cell 3：工具函数

In [8]:
# ===== Cell 3: 工具函数（最终稳定版，不再使用 scanpy HVG） =====

def split_items(x):
    return [i.strip() for i in str(x).split(",") if i.strip()]

def normalize_pathway_name(x):
    return x.strip().replace("-", "_").replace(" ", "_")

def get_valid_genes(adata, genes):
    return [g for g in genes if g in adata.var_names]

def sparse_or_dense_mean_by_genes(adata, genes):
    """
    计算一组基因在每个 spot/cell 中的平均表达。
    支持 sparse / dense。
    """
    valid_genes = get_valid_genes(adata, genes)
    if len(valid_genes) == 0:
        return np.zeros(adata.n_obs, dtype=float)

    X = adata[:, valid_genes].X
    if sp.issparse(X):
        return np.asarray(X.mean(axis=1)).reshape(-1)
    return np.asarray(X).mean(axis=1).reshape(-1)

def minmax_1d(x):
    x = np.asarray(x).reshape(-1, 1)
    return MinMaxScaler().fit_transform(x).reshape(-1)

def build_metabolic_net_from_quartets(tmcn_df, adata):
    """
    把四元组中的 Source_Genes 构造成 AUCell 需要的网络格式。
    """
    rows = []
    for _, row in tmcn_df.iterrows():
        axis = row["TMCN_Name"].replace("TMCN_", "").replace("_Axis", "")
        for g in split_items(row["Source_Genes"]):
            if g in adata.var_names:
                rows.append({"source": axis, "target": g})
    return pd.DataFrame(rows)

def collect_forced_genes(tmcn_df):
    """
    收集必须保留的四元组基因：
    Source_Genes ∪ Target_Genes
    """
    forced = set()
    for _, row in tmcn_df.iterrows():
        forced.update(split_items(row["Source_Genes"]))
        forced.update(split_items(row["Target_Genes"]))
    return forced

def compute_tmcn_scores(adata_full, tmcn_df):
    """
    阶段 1：在全量 adata 上计算：
    1. PROGENy pathway activity
    2. AUCell machinery activity
    3. Sender / Receiver 综合得分
    """
    # ---------- 步骤 1.1：PROGENy ----------
    net_progeny = dc.get_progeny(organism="human", top=500)
    dc.run_mlm(
        mat=adata_full,
        net=net_progeny,
        source="source",
        target="target",
        weight="weight",
        verbose=False,
        use_raw=False,
    )

    df_A = dc.get_acts(adata_full, obsm_key="mlm_estimate").to_df()
    df_A.columns = [f"PROGENy_{normalize_pathway_name(c)}" for c in df_A.columns]

    # ---------- 步骤 1.2：AUCell ----------
    metabolic_net = build_metabolic_net_from_quartets(tmcn_df, adata_full)
    dc.run_aucell(
        mat=adata_full,
        net=metabolic_net,
        source="source",
        target="target",
        min_n=1,
        verbose=False,
        use_raw=False,
    )

    df_B = dc.get_acts(adata_full, obsm_key="aucell_estimate").to_df()
    df_B.columns = [f"AUCell_{c}" for c in df_B.columns]

    # ---------- 步骤 1.3：归一化 ----------
    df_A_scaled = pd.DataFrame(
        MinMaxScaler().fit_transform(df_A),
        index=df_A.index,
        columns=df_A.columns
    )

    df_B_scaled = pd.DataFrame(
        MinMaxScaler().fit_transform(df_B),
        index=df_B.index,
        columns=df_B.columns
    )

    # ---------- 计算 Sender / Receiver ----------
    for _, row in tmcn_df.iterrows():
        axis = row["TMCN_Name"].replace("TMCN_", "").replace("_Axis", "")
        source_pathways = [normalize_pathway_name(x) for x in split_items(row["Source_Pathways"])]
        target_genes = split_items(row["Target_Genes"])

        progeny_cols = [f"PROGENy_{p}" for p in source_pathways if f"PROGENy_{p}" in df_A_scaled.columns]
        aucell_col = f"AUCell_{axis}"

        if len(progeny_cols) == 0:
            raise ValueError(f"{axis}: 没找到 PROGENy 列，请检查 Source_Pathways 名称。")

        if aucell_col not in df_B_scaled.columns:
            raise ValueError(f"{axis}: 没找到 AUCell 列 {aucell_col}。")

        sender_score = df_A_scaled[progeny_cols].mean(axis=1) * df_B_scaled[aucell_col]
        receiver_score = minmax_1d(sparse_or_dense_mean_by_genes(adata_full, target_genes))

        adata_full.obs[f"{axis}_Sender_Score"] = sender_score.values
        adata_full.obs[f"{axis}_Receiver_Score"] = receiver_score

    return adata_full, df_A, df_B, df_A_scaled, df_B_scaled


# =========================
# 新增：稳定的矩阵清洗函数
# =========================
def sanitize_matrix(mat):
    """
    把矩阵中的 nan / inf 清理掉。
    """
    if sp.issparse(mat):
        mat = mat.copy().tocsr()
        mat.data = np.nan_to_num(mat.data, nan=0.0, posinf=0.0, neginf=0.0)
        return mat
    else:
        return np.nan_to_num(np.asarray(mat), nan=0.0, posinf=0.0, neginf=0.0)

def get_matrix_for_hvg(adata):
    """
    优先使用 counts layer；
    如果没有，就使用 X。
    返回清洗后的矩阵和来源名字。
    """
    if "counts" in adata.layers:
        X = sanitize_matrix(adata.layers["counts"])
        source_name = "counts"
    else:
        X = sanitize_matrix(adata.X)
        source_name = "X"
    return X, source_name

def compute_gene_dispersion_manual(X):
    """
    手动计算每个基因的离散度（dispersion），避免 scanpy HVG 的 binning 报错。
    dispersion = variance / mean
    """
    if sp.issparse(X):
        X = X.tocsr()
        gene_mean = np.asarray(X.mean(axis=0)).reshape(-1)
        gene_mean_sq = np.asarray(X.multiply(X).mean(axis=0)).reshape(-1)
        gene_var = gene_mean_sq - gene_mean ** 2
    else:
        gene_mean = np.mean(X, axis=0)
        gene_var = np.var(X, axis=0)

    gene_mean = np.nan_to_num(gene_mean, nan=0.0, posinf=0.0, neginf=0.0)
    gene_var = np.nan_to_num(gene_var, nan=0.0, posinf=0.0, neginf=0.0)

    # 防止负数方差（数值误差）
    gene_var = np.maximum(gene_var, 0.0)

    # dispersion = variance / mean
    dispersion = gene_var / (gene_mean + 1e-8)
    dispersion = np.nan_to_num(dispersion, nan=0.0, posinf=0.0, neginf=0.0)

    return gene_mean, gene_var, dispersion

def select_top_hvg_manual(adata, n_top_genes=2000):
    """
    手动选择 HVG：
    - 不使用 scanpy.pp.highly_variable_genes
    - 直接按 dispersion 排名前 n_top_genes
    """
    X_hvg, source_name = get_matrix_for_hvg(adata)
    gene_mean, gene_var, dispersion = compute_gene_dispersion_manual(X_hvg)

    df_stats = pd.DataFrame({
        "gene": adata.var_names,
        "mean": gene_mean,
        "var": gene_var,
        "dispersion": dispersion
    })

    # 去掉完全无信息的基因
    df_stats = df_stats[(df_stats["mean"] >= 0) & np.isfinite(df_stats["dispersion"])].copy()

    # 按 dispersion 从大到小选前 n_top_genes
    df_stats = df_stats.sort_values("dispersion", ascending=False)
    hvg_genes = df_stats["gene"].head(n_top_genes).tolist()

    print(f"HVG 来源矩阵: {source_name}")
    print(f"可用于排序的基因数: {df_stats.shape[0]}")
    print(f"选出的 HVG 数: {len(hvg_genes)}")

    return set(hvg_genes), df_stats

def subset_hvg_plus_quartet_genes(adata_full, tmcn_df, n_top_genes=2000):
    """
    训练用基因子集：
    HVG(2000) ∪ 四元组基因
    """
    forced_genes = collect_forced_genes(tmcn_df)
    forced_genes = {g for g in forced_genes if g in adata_full.var_names}

    hvg_genes, df_hvg_stats = select_top_hvg_manual(adata_full, n_top_genes=n_top_genes)

    keep_genes = sorted(hvg_genes.union(forced_genes))
    adata_model = adata_full[:, keep_genes].copy()

    # 再清一次，确保模型输入无坏值
    adata_model.X = sanitize_matrix(adata_model.X)
    if "counts" in adata_model.layers:
        adata_model.layers["counts"] = sanitize_matrix(adata_model.layers["counts"])

    print(f"原始基因数: {adata_full.n_vars}")
    print(f"手动HVG数: {len(hvg_genes)}")
    print(f"强制保留四元组基因数: {len(forced_genes)}")
    print(f"最终训练基因数: {adata_model.n_vars}")

    if sp.issparse(adata_model.X):
        n_bad_x = np.sum(~np.isfinite(adata_model.X.data))
    else:
        n_bad_x = np.sum(~np.isfinite(np.asarray(adata_model.X)))
    print(f"adata_model.X 中非有限值个数: {int(n_bad_x)}")

    if "counts" in adata_model.layers:
        if sp.issparse(adata_model.layers["counts"]):
            n_bad_counts = np.sum(~np.isfinite(adata_model.layers["counts"].data))
        else:
            n_bad_counts = np.sum(~np.isfinite(np.asarray(adata_model.layers["counts"])))
        print(f"adata_model.layers['counts'] 中非有限值个数: {int(n_bad_counts)}")

    return adata_model, keep_genes, df_hvg_stats

def build_tmcn_graph(adata_model, axes, n_neighs=8):
    """
    阶段 2.2：构建代谢加权空间图
    """
    sq.gr.spatial_neighbors(
        adata_model,
        coord_type="generic",
        spatial_key="spatial",
        n_neighs=n_neighs
    )

    spatial_adj = adata_model.obsp["spatial_connectivities"].tocsr()

    S = np.vstack([adata_model.obs[f"{axis}_Sender_Score"].to_numpy() for axis in axes]).T
    R = np.vstack([adata_model.obs[f"{axis}_Receiver_Score"].to_numpy() for axis in axes]).T

    row_idx, col_idx = spatial_adj.nonzero()
    base_weights = np.asarray(spatial_adj[row_idx, col_idx]).reshape(-1)

    metabolic_affinity = np.sum(S[row_idx] * R[col_idx], axis=1)
    edge_weights = np.clip(base_weights * metabolic_affinity, a_min=1e-6, a_max=None)

    tmcn_adj = sp.csr_matrix(
        (edge_weights, (row_idx, col_idx)),
        shape=spatial_adj.shape
    )

    adata_model.obsp["tmcn_connectivities"] = tmcn_adj.maximum(tmcn_adj.T)
    return adata_model

def load_or_download_nichenet_cache(cache_dir):
    """
    读取或下载 NicheNet 先验，并缓存到本地。
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    lr_path = cache_dir / "nichenet_lr_network.csv"
    lt_path = cache_dir / "nichenet_ligand_target_matrix.csv"

    if lr_path.exists() and lt_path.exists():
        print("从本地缓存加载 NicheNet...")
        gp_dict = nc.utils.extract_gp_dict_from_nichenet_lrt_interactions(
            species="human",
            version="v2",
            keep_target_genes_ratio=0.25,
            max_n_target_genes_per_gp=50,
            load_from_disk=True,
            lr_network_file_path=str(lr_path),
            ligand_target_matrix_file_path=str(lt_path),
            plot_gp_gene_count_distributions=False,
        )
    else:
        print("首次下载并缓存 NicheNet...")
        gp_dict = nc.utils.extract_gp_dict_from_nichenet_lrt_interactions(
            species="human",
            version="v2",
            keep_target_genes_ratio=0.25,
            max_n_target_genes_per_gp=50,
            save_to_disk=True,
            lr_network_file_path=str(lr_path),
            ligand_target_matrix_file_path=str(lt_path),
            plot_gp_gene_count_distributions=False,
        )
    return gp_dict

def annotate_tmcn_clusters(adata_model, axes, cluster_key="TMCN_Clusters", threshold=0.05):
    """
    阶段 3.2：根据 Sender*Receiver 强度给 cluster 打标签
    """
    rows = []
    for cluster in adata_model.obs[cluster_key].cat.categories:
        mask = adata_model.obs[cluster_key] == cluster
        row = {"Cluster": cluster}
        for axis in axes:
            intensity = (
                adata_model.obs.loc[mask, f"{axis}_Sender_Score"] *
                adata_model.obs.loc[mask, f"{axis}_Receiver_Score"]
            ).mean()
            row[f"{axis}_Intensity"] = intensity
        rows.append(row)

    df_cluster = pd.DataFrame(rows).set_index("Cluster")

    def label_cluster(row):
        best_axis = row.idxmax().replace("_Intensity", "")
        best_val = row.max()
        if best_val > threshold:
            return f"TMCN_{best_axis}_Axis"
        return "Background/Other_Niche"

    df_cluster["TMCN_Annotation"] = df_cluster.apply(label_cluster, axis=1)
    adata_model.obs["TMCN_Niche"] = adata_model.obs[cluster_key].map(
        df_cluster["TMCN_Annotation"].to_dict()
    )
    return adata_model, df_cluster

Cell 4：阶段 1，全量打分

In [9]:
# ===== Cell 4: 阶段1 - 全量打分 =====
file_path = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
adata_full = sc.read_h5ad(file_path)

print(adata_full)
print("has counts layer:", "counts" in adata_full.layers)
print("has spatial coords:", "spatial" in adata_full.obsm)

adata_full, df_A, df_B, df_A_scaled, df_B_scaled = compute_tmcn_scores(adata_full, tmcn_df)

score_cols = [c for c in adata_full.obs.columns if c.endswith("_Score")]
display(adata_full.obs[score_cols].describe().T)

AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'old_annot_type', 'old_fine_annot_type', '1', '2', '3', '4', '5', 'scaled_x', 'scaled_y', 'fine_annot', 'annot', 'fine_annot_type', 'ground_truth', 'annot_type'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'
has counts layer: False
has spatial coords: True


,count,mean,std,min,25%,50%,75%,max
Lactate_Sender_Score,3798.0,0.127916,0.074464,0.0,0.089275,0.122083,0.160193,0.557291
Lactate_Receiver_Score,3798.0,0.100777,0.138558,0.0,0.000000,0.000000,0.125000,1.000000
Adenosine_Sender_Score,3798.0,0.020091,0.037005,0.0,0.000000,0.000000,0.030086,0.377811
Adenosine_Receiver_Score,3798.0,0.169061,0.162829,0.0,0.047619,0.142857,0.238095,1.000000
PGE2_Sender_Score,3798.0,0.023789,0.076771,0.0,0.000000,0.000000,0.000000,0.569351
PGE2_Receiver_Score,3798.0,0.059110,0.130944,0.0,0.000000,0.000000,0.000000,1.000000
Glutamine_Sender_Score,3798.0,0.096796,0.057907,0.0,0.060777,0.094363,0.129142,0.384487
Glutamine_Receiver_Score,3798.0,0.165751,0.133950,0.0,0.071429,0.142857,0.238095,1.000000
Succinate_Sender_Score,3798.0,0.007008,0.026770,0.0,0.000000,0.000000,0.000000,0.381174
Succinate_Receiver_Score,3798.0,0.012375,0.068375,0.0,0.000000,0.000000,0.000000,1.000000


Cell 5：构建练用 adata_model（2000 HVG + 四元组基因）

In [10]:
# ===== Cell 5: HVG + 四元组基因子集 =====
adata_model, keep_genes, df_hvg_stats = subset_hvg_plus_quartet_genes(
    adata_full,
    tmcn_df,
    n_top_genes=2000
)

print(adata_model)
display(df_hvg_stats.head(20))

HVG 来源矩阵: X
可用于排序的基因数: 36601
选出的 HVG 数: 2000
原始基因数: 36601
手动HVG数: 2000
强制保留四元组基因数: 25
最终训练基因数: 2017
adata_model.X 中非有限值个数: 0
AnnData object with n_obs × n_vars = 3798 × 2017
    obs: 'in_tissue', 'array_row', 'array_col', 'old_annot_type', 'old_fine_annot_type', '1', '2', '3', '4', '5', 'scaled_x', 'scaled_y', 'fine_annot', 'annot', 'fine_annot_type', 'ground_truth', 'annot_type', 'Lactate_Sender_Score', 'Lactate_Receiver_Score', 'Adenosine_Sender_Score', 'Adenosine_Receiver_Score', 'PGE2_Sender_Score', 'PGE2_Receiver_Score', 'Glutamine_Sender_Score', 'Glutamine_Receiver_Score', 'Succinate_Sender_Score', 'Succinate_Receiver_Score'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial', 'mlm_estimate', 'mlm_pvals', 'aucell_estimate'


,gene,mean,var,dispersion
4377,IGKC,111.126259,79029.382812,711.167480
21317,MGP,127.520958,81610.039062,639.973572
34659,IGLC2,32.381744,17532.427734,541.429382
24993,IGHG3,58.340298,27786.529297,476.283630
15734,COX6C,269.692017,77080.750000,285.810272
34669,IGLC7,0.871770,236.458908,271.240112
8462,ALB,4.638986,1111.861572,239.677734
24990,IGHA1,10.114775,1844.468994,182.353943
34657,IGLC1,8.480238,1482.918701,174.867584
34661,IGLC3,4.254847,730.242676,171.626083


Cell 6：阶段 2，构建代谢加权图 + 初始化模型

In [13]:
# ===== Cell 6: 阶段2 - 图构建 + 初始化模型 =====
adata_model = build_tmcn_graph(adata_model, axes=AXES, n_neighs=8)
print("tmcn_connectivities built:", adata_model.obsp["tmcn_connectivities"])

node_feature_keys = []
for axis in AXES:
    node_feature_keys.append(f"{axis}_Sender_Score")
    node_feature_keys.append(f"{axis}_Receiver_Score")
print("node_feature_keys =", node_feature_keys)

cache_dir = "/home/zhangjunyi/xiangmu/nichecompass-main/cache_nichenet"


tmcn_connectivities built:   (0, 397)	0.0024699349887669086
  (0, 487)	0.012349674478173256
  (0, 923)	0.000929095025639981
  (0, 1087)	0.004939869977533817
  (0, 2195)	1e-06
  (0, 2425)	0.07423961162567139
  (0, 3242)	0.01975947991013527
  (0, 3696)	0.0024699349887669086
  (0, 3759)	0.007409804966300726
  (1, 588)	0.2983357310295105
  (1, 1023)	0.21116438508033752
  (1, 1773)	0.2116738259792328
  (1, 1911)	0.18581362068653107
  (1, 2367)	0.24033047258853912
  (1, 2633)	0.10783448815345764
  (1, 2743)	0.21797437965869904
  (1, 3165)	0.27754345536231995
  (1, 3464)	0.20686858892440796
  (2, 222)	0.09516807645559311
  (2, 1092)	0.022671323269605637
  (2, 1326)	1e-06
  (2, 2157)	0.01662595197558403
  (2, 2321)	0.021161159500479698
  (2, 3105)	0.03374463692307472
  (2, 3107)	0.0052902898751199245
  :	:
  (3795, 67)	0.17588013410568237
  (3795, 301)	0.09984132647514343
  (3795, 859)	0.0731029361486435
  (3795, 1769)	0.13731685280799866
  (3795, 1945)	0.167686328291893
  (3795, 2278)	0.11869

In [14]:
gp_dict = load_or_download_nichenet_cache(cache_dir)



首次下载并缓存 NicheNet...


In [ ]:
nc.utils.add_gps_from_gp_dict_to_adata(
    gp_dict=gp_dict,
    adata=adata_model,
)

FAST_DEBUG = True   # 先调试，再正式跑

if FAST_DEBUG:
    n_addon_gp = 0
    n_epochs = 1
    n_epochs_all_gps = 1
    edge_batch_size = 64
    node_batch_size = 128
else:
    n_addon_gp = 0
    n_epochs = 50
    n_epochs_all_gps = 10
    edge_batch_size = 128
    node_batch_size = 256

model = nc.models.NicheCompass(
    adata=adata_model,
    counts_key="counts" if "counts" in adata_model.layers else None,
    adj_key="tmcn_connectivities",
    conv_layer_encoder="gcnconv",
    node_feature_keys=node_feature_keys,
    n_addon_gp=n_addon_gp,
)

print("torch.cuda.is_available() =", torch.cuda.is_available())
print("model device before train =", next(model.model.parameters()).device)

Cell 7：阶段 2.3，模型训练

In [ ]:
# ===== Cell 7: 模型训练 =====
# ===== 训练前检查 =====
print("训练前 model 参数所在设备：", next(model.model.parameters()).device)
print("torch.cuda.is_available() =", torch.cuda.is_available())

# ===== 开始训练 =====
model.train(
    n_epochs=n_epochs,
    n_epochs_all_gps=n_epochs_all_gps,
    edge_batch_size=edge_batch_size,
    node_batch_size=node_batch_size,
    use_cuda_if_available=True,   # 这里直接写 True
)

Cell 8：阶段 3，latent 聚类与生态位注释

In [ ]:
# ===== Cell 8: latent 聚类与 TMCN 注释 =====
adata_model.obsm["tmcn_latent"] = model.get_latent_representation(
    node_batch_size=128 if FAST_DEBUG else 256
)

sc.pp.neighbors(
    adata_model,
    use_rep="tmcn_latent",
    key_added="tmcn_latent_neighbors"
)

sc.tl.leiden(
    adata_model,
    neighbors_key="tmcn_latent_neighbors",
    key_added="TMCN_Clusters",
    resolution=0.8
)

adata_model, df_cluster_scores = annotate_tmcn_clusters(
    adata_model,
    axes=AXES,
    cluster_key="TMCN_Clusters",
    threshold=0.05
)

display(df_cluster_scores)
print("cluster number:", adata_model.obs["TMCN_Clusters"].nunique())
print(adata_model.obs["TMCN_Niche"].value_counts())

Cell 9：空间可视化 + 保存结果

In [ ]:
# ===== Cell 9: 空间可视化 + 保存结果 =====
plt.rcParams["figure.figsize"] = (8, 8)

if "spatial" in adata_model.uns:
    sc.pl.spatial(
        adata_model,
        color="TMCN_Niche",
        spot_size=30,
        title="Tumor-Metabolic Communication Niches (TMCN)",
        show=True,
    )
else:
    sq.pl.spatial_scatter(
        adata_model,
        color="TMCN_Niche",
        size=30,
    )
    plt.show()

save_path = file_path.replace(".h5ad", "_TMCN_HVG2000_FinalAnnotated.h5ad")
adata_model.write_h5ad(save_path)
print("结果保存到：", save_path)